# NLP + Transformers
**Laboratorio 07 – Inspección de Arquitectura, Hidden States y Autoatención en BERT (BETO)**

Modelo: `dccuchile/bert-base-spanish-wwm-cased`

---
## 1) Visualización de la Configuración de BERT 👀

In [5]:
%pip install transformers torch

In [6]:
from transformers import BertConfig, BertModel

Cargar la configuración del modelo BERT en español

In [7]:
config = BertConfig.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
print("Número de capas de BERT:", config.num_hidden_layers)

Número de capas de BERT: 12


Cargar el modelo para visualizar su arquitectura

In [8]:
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)
print(model)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [9]:
print("hidden_size:", config.hidden_size)
print("num_attention_heads:", config.num_attention_heads)
print("intermediate_size:", config.intermediate_size)
print("vocab_size:", config.vocab_size)

hidden_size: 768
num_attention_heads: 12
intermediate_size: 3072
vocab_size: 31002


### Análisis – Sección 1

**1. Los tres módulos principales de `BertModel`**

- **`embeddings` (`BertEmbeddings`)**: convierte los ids de los tokens en vectores de dimensión 768. Suma tres tablas de embeddings: `word_embeddings` (vocabulario × 768, ~31 002 tokens), `position_embeddings` (hasta 512 posiciones × 768) y `token_type_embeddings` (2 segmentos × 768). Luego aplica `LayerNorm` y `Dropout`.
- **`encoder` (`BertEncoder`)**: es una `ModuleList` de **12 bloques `BertLayer`** apilados. Cada bloque refina la representación de cada token con atención multi-cabeza (12 cabezas) y una red feed-forward.
- **`pooler` (`BertPooler`)**: toma el vector del primer token (`[CLS]`) de la última capa y le aplica una capa `Linear(768, 768)` seguida de `Tanh`. Produce un vector resumen de toda la oración (`pooler_output`), útil para clasificación.

**2. Subcapas dentro de una `BertLayer`**

- **Atención (`attention.self` → `BertSelfAttention`)**: tres proyecciones lineales `query`, `key` y `value`, cada una `Linear(768 → 768)`, más un `dropout` sobre los pesos de atención. Los 768 se reparten en 12 cabezas de 64 dimensiones.
- **`attention.output` (`BertSelfOutput`)**: `dense` (`Linear 768→768`, proyección de salida de las cabezas concatenadas), `LayerNorm` y `Dropout`. Aquí ocurre la **conexión residual + normalización**: `LayerNorm(x + dropout(dense(contexto)))`.
- **`intermediate` (`BertIntermediate`)**: `Linear(768 → 3072)` + activación GELU (primera parte del feed-forward).
- **`output` (`BertOutput`)**: `Linear(3072 → 768)`, `Dropout` y `LayerNorm`; segunda **conexión residual + normalización**: `LayerNorm(x + dropout(dense(h)))`.

Por tanto, hay **dos `LayerNorm` por capa** (uno tras la atención y otro tras el feed-forward), más el `LayerNorm` de `embeddings`.

**3. Dimensión oculta**

`hidden_size` = **d_model = 768**, visible en las capas lineales `Linear(in_features=768, out_features=768)` de query/key/value, en los embeddings y en el pooler. La capa intermedia del feed-forward se expande a 3072 (4 × 768) y vuelve a 768.

## 2) Visualización de las Capas y Hidden States 🔍

In [10]:
from transformers import BertTokenizer
import torch

Cargar tokenizer y modelo BERT en español

In [11]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Texto de ejemplo

In [12]:
text = "El producto de Samsung Galaxy S21 llegó el 12 de marzo y superó mis expectativas."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
outputs = model(**inputs)

In [13]:
print(inputs)
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

{'input_ids': tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
['[CLS]', 'El', 'producto', 'de', 'Sam', '##su', '##ng', 'Gala', '##xy', 'S', '##21', 'llegó', 'el', '12', 'de', 'marzo', 'y', 'superó', 'mis', 'expectativas', '.', '[SEP]']


In [14]:
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3688, -0.2218, -0.6865,  ..., -0.4430,  0.2883,  0.1794],
         [-0.2616, -0.0662, -0.6243,  ..., -0.5904,  0.6507,  0.2768],
         [-0.2415, -0.2261, -0.6785,  ..., -0.1096,  0.0910,  0.5395],
         ...,
         [-1.4393, -0.1986, -0.5770,  ..., -0.5261,  0.0716,  1.0376],
         [-0.7026,  0.3509, -0.4953,  ..., -0.3322,  0.3734,  0.1284],
         [-1.2273, -0.4597, -1.0296,  ..., -0.6864,  0.9237, -0.5841]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 1.9538e-01, -2.7669e-01, -3.8370e-01, -2.6586e-01, -2.3251e-02,
         -1.6733e-01, -4.1082e-01, -3.3960e-02,  2.9060e-01,  5.0418e-01,
         -7.2097e-01,  2.6635e-01,  4.6437e-01, -4.5988e-01,  2.9262e-01,
         -2.2400e-01, -1.1267e-01, -1.2723e-01,  7.3343e-02,  4.4665e-01,
         -4.4931e-01,  7.0735e-01,  4.4936e-01, -1.9415e-01,  1.6039e-02,
          6.1270e-01, -3.4631e-01,  1.4232e-01, -1.2409e-01, -3.097

Extraer los hidden states

In [15]:
hidden_states = outputs.hidden_states
print("Número de capas (incluyendo la capa de embedding):", len(hidden_states))

Número de capas (incluyendo la capa de embedding): 13


Por ejemplo, si BERT base tiene 12 capas, se imprimirá 13 (1 de embedding + 12 codificadores)

In [16]:
print("Shape de hidden_states[0]:", hidden_states[0].shape)
print("Shape de hidden_states[-1]:", hidden_states[-1].shape)
print("Número de tokens:", inputs["input_ids"].shape[1])

Shape de hidden_states[0]: torch.Size([1, 22, 768])
Shape de hidden_states[-1]: torch.Size([1, 22, 768])
Número de tokens: 22


### Análisis – Sección 2

**1. Diccionario `inputs`**

- **`input_ids`**: tensor con los ids enteros de cada token según el vocabulario de BETO. Incluye los tokens especiales `[CLS]` al inicio y `[SEP]` al final. El texto se divide en subpalabras (WordPiece): términos poco frecuentes como "Samsung", "Galaxy" o "S21" pueden partirse en varios fragmentos (`##...`). Estos ids son los índices que consulta `word_embeddings`.
- **`token_type_ids`**: indica a qué segmento pertenece cada token (0 = primera oración, 1 = segunda). Se usa en tareas con pares de oraciones (NSP, inferencia, QA). Como aquí hay una sola oración, **todos son 0**; el modelo suma el embedding de tipo 0 a cada token.
- **`attention_mask`**: indica qué posiciones son tokens reales (1) y cuáles son *padding* (0), para que la atención ignore el relleno. Está compuesta solo por **1** porque procesamos **una única oración sin padding**: todos los tokens son reales.

**2. ¿Por qué `len(hidden_states)` = 13 si hay 12 capas?**

La tupla `hidden_states` contiene **13 tensores**: el de la posición **0** es la **salida de la capa de embeddings** (embedding de palabra + posición + tipo de segmento, tras `LayerNorm` y dropout), es decir, la entrada del primer bloque codificador; las posiciones **1 a 12** son las salidas de cada uno de los 12 `BertLayer`. `config.num_hidden_layers` solo cuenta los bloques del encoder, no la capa de embeddings. La posición 12 (`hidden_states[-1]`) coincide con `last_hidden_state`.

**3. `hidden_states[0].shape` → `[batch_size, sequence_length, hidden_size]`**

- **`batch_size` = 1**: cantidad de oraciones procesadas a la vez (aquí solo una).
- **`sequence_length`** = número de tokens tras la tokenización, incluyendo `[CLS]` y `[SEP]` (es el valor impreso como "Número de tokens" en la celda anterior). Es mayor que el número de palabras por las subpalabras.
- **`hidden_size` = 768**: dimensión del vector que representa a cada token (d_model).

Cada token tiene su propio vector de 768 números; la forma se conserva en las 13 salidas, y lo que cambia capa a capa es el contenido: de representaciones léxicas (embeddings) a representaciones contextuales.

## 3) Diagrama Conceptual y Pseudocódigo 🙌

Pseudocódigo del mecanismo de Self-Attention

### Pseudocódigo de Self-Attention (Scaled Dot-Product, multi-cabeza)

Fórmula: **Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V**

```text
ENTRADA: X  de forma [seq_len, d_model]     (d_model = 768)
         pesos Wq, Wk, Wv de forma [d_model, d_model]
         h = 12 cabezas, d_k = d_model / h = 64
         mask (attention_mask)

1. Proyección lineal de cada token:
      Q = X · Wq + bq        # [seq_len, 768]  "¿qué busco?"
      K = X · Wk + bk        # [seq_len, 768]  "¿qué ofrezco?"
      V = X · Wv + bv        # [seq_len, 768]  "¿qué información llevo?"

2. Dividir en h cabezas:
      Q, K, V → [h, seq_len, d_k]

3. PARA cada cabeza i = 1..h:
      a) scores_i  = Q_i · K_iᵀ              # [seq_len, seq_len], similitud entre pares de tokens
      b) scores_i  = scores_i / sqrt(d_k)    # escalado: evita softmax saturado
      c) scores_i  = scores_i + mask         # padding → -inf (peso 0 tras softmax)
      d) pesos_i   = softmax(scores_i, por fila)   # cada fila suma 1
      e) pesos_i   = dropout(pesos_i)
      f) contexto_i = pesos_i · V_i          # suma ponderada de los valores [seq_len, d_k]

4. Concatenar las h cabezas:
      contexto = concat(contexto_1..contexto_h)   # [seq_len, 768]

5. Proyección de salida, residual y normalización (BertSelfOutput):
      Z = LayerNorm(X + dropout(contexto · Wo + bo))

SALIDA: Z de forma [seq_len, d_model]
```

**Explicación de cada operación**

- **Proyección lineal**: cada token se transforma en tres vectores distintos (consulta, clave, valor) para que el modelo aprenda roles diferentes de la misma representación.
- **Q·Kᵀ**: el producto punto mide cuánto "atiende" cada token a todos los demás; da una matriz `seq_len × seq_len`.
- **División por √d_k**: mantiene la varianza de los scores estable para que el softmax no se sature y los gradientes sean útiles.
- **Máscara**: las posiciones de padding reciben −∞ para que su peso sea 0.
- **Softmax**: convierte los scores en probabilidades (pesos de atención) por fila.
- **Suma ponderada con V**: el nuevo vector de cada token es la combinación de los valores de todos los tokens, ponderada por esos pesos; así incorpora contexto bidireccional.
- **Multi-cabeza**: 12 cabezas de 64 dimensiones permiten atender a distintos tipos de relaciones (sintácticas, semánticas, posicionales) en paralelo.

### ¿Por qué el diagrama y el pseudocódigo van al cierre y cómo se relacionan con `BertSelfAttention`?

Se ubican **al final** porque primero hay que *observar* la arquitectura real (Sección 1: `print(model)`) y el flujo de tensores (Sección 2: hidden states y sus shapes). Con esa evidencia empírica, el pseudocódigo sirve para **explicar qué ocurre dentro** de cada bloque: pasa de la descripción de "qué módulos existen" al "cómo calculan".

La relación con las capas impresas en la Sección 1 es directa:

| Pseudocódigo | Módulo en `print(model)` |
|---|---|
| Paso 1: Q, K, V | `BertSelfAttention`: `query`, `key`, `value` (`Linear 768→768`) |
| Pasos 2–3: cabezas, softmax, dropout, suma con V | Operaciones internas de `BertSelfAttention` (`dropout` incluido; 12 cabezas × 64 dims) |
| Paso 5: proyección de salida + residual + LayerNorm | `BertSelfOutput`: `dense`, `LayerNorm`, `Dropout` |

Es decir, las tres `Linear` que vimos impresas son exactamente las proyecciones Wq, Wk y Wv del pseudocódigo, y los pasos de producto punto, escalado y softmax no tienen parámetros, por eso no aparecen como capas en `print(model)`. Las 12 `BertLayer` repiten este cálculo, por lo que los `hidden_states` de la Sección 2 son la salida de aplicarlo capa tras capa.